## Import lib

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader , TensorDataset
from torchvision.utils import save_image, make_grid
from torch.optim import Adam
import torch.nn.init as init

import numpy as np
import math

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import MultipleLocator
import matplotlib.cm as cm

import copy
import seaborn as sns

from scipy.stats import norm
from sklearn.neighbors import KernelDensity, LocalOutlierFactor
from sklearn.decomposition import PCA
from sklearn.datasets import make_regression
from sklearn.feature_selection import mutual_info_regression

from matplotlib.ticker import StrMethodFormatter

import pandas as pd
import tqdm

import pickle

import os
import sys
# MI estimators
from utils.estimators import *

## Load data

In [ ]:
NUM_POLICY = 3

# Load flat/rough/cut weight dataset
terrain = {"flat" :[], "rough":[] , "weight":[]}
for i in terrain.keys(): # each run have 3 policies
    for policy in range(NUM_POLICY):
        terrain[i].append(np.load(f"data/eval/new_pol2/{i}-policy_{policy}.npy" , allow_pickle=True).item())
for key, value in terrain.items():
    terrain[key] = np.array(value)

# Load sensory albation dataset
sensor_loss = {"full":[] , "no_pos":[] , "no_vel":[], "no_action":[], "no_IMU":[] , "no_fc":[]}
for i,keys in enumerate(sensor_loss.keys()):
    for policy in range(NUM_POLICY):    
        sensor_loss[keys].append(np.load(f"data/eval/new_pol2/case_{i}-policy_{policy}.npy" , allow_pickle=True).item())
for key, value in sensor_loss.items():
    sensor_loss[key] = np.array(value)

# [case:str][number_file:int][data/brain:str]

In [ ]:
selected_dataset = terrain

Format will be

- [`key`:case][`int`:policy_n][`key`:data group][`key`:data inside]

Performance measuring

In [ ]:

performance = {f"policy{n_policy}": {case: [] for case in selected_dataset.keys()} for n_policy in range(3)}
max_distance = {f"policy{n_policy}": {case: [] for case in selected_dataset.keys()} for n_policy in range(3)}
avg_vel = {f"policy{n_policy}": {case: [] for case in selected_dataset.keys()} for n_policy in range(3)}

for policy in range(NUM_POLICY):
    for case in selected_dataset.keys():
        reward = np.sum(selected_dataset[case][policy]["data"]['reward'], axis=0)
        performance[f"policy{policy}"][case] = reward

        max_pos_x = np.max(selected_dataset[case][policy]["data"]['pos_x'], axis=0)
        max_distance[f"policy{policy}"][case] = max_pos_x

        vel_x = selected_dataset[case][policy]["data"]['vel_b'][:,:,0]
        avg_vel[f"policy{policy}"][case] = np.mean(vel_x ,axis=0)

print(performance.keys())  # Prints the policy keys
print(performance["policy0"].keys())  # Prints the case keys under "policy0"
print(max_distance.keys())  # Prints the policy keys
print(max_distance["policy0"].keys())  # Prints the case keys under "policy0"
# [num_policy][data group][data inside]


## Evaluate

In [ ]:
def plot_performance(performance , title:str="Performance"):
    num_col = len(performance.keys())  # Number of policies (columns)
    num_group = len(performance['policy0'].keys())  # Number of cases (groups)
    fig, axes = plt.subplots(1, num_col, figsize=(12, 6))
    
    pastel_colors = sns.color_palette("pastel", n_colors=num_group)
    case_colors = {case: pastel_colors[i] for i, case in enumerate(list(performance['policy0'].keys()))}
    
    for i, policy in enumerate(performance.keys()):
        ax = axes[i]  # Get the axis for this policy subplot
        box = ax.boxplot(
            [performance[policy][case] for case in performance[policy].keys()],  # List of data for each case
                vert=True, patch_artist=True,zorder=1  # fill with color  # Set the tick labels to be the case names
        )
        ax.set_xticklabels(list(performance[policy].keys()), fontsize=8) 
        for j, patch in enumerate(box['boxes']):
            case = list(performance[policy].keys())[j]  # Get the case corresponding to the current box
            patch.set_facecolor(case_colors[case])  # Set the color for the box based on the case

        for j, case in enumerate(performance[policy].keys()):
            x_vals = [j + 1 + np.random.normal(0, 0.1) for _ in performance[policy][case]]  # Jitter with normal noise
            ax.scatter(x_vals, performance[policy][case], alpha=0.8, s=5, label=case, zorder=2)
        ax.set_title(f"{title} for {policy}")
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_trajectory(dataset , policy:int):
    num_cases = len(dataset.keys())  # Number of cases in the dataset
    # Calculate the number of rows and columns for a square-like grid
    num_cols = int(np.ceil(np.sqrt(num_cases)))  # Number of columns
    num_rows = int(np.ceil(num_cases / num_cols))  # Number of rows to fit all subplots
    
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(18, 16))  # Create a grid of subplots
    axes = axes.flatten()  # Flatten the axes array to make indexing easier
    
    for i, (case, ax) in enumerate(zip(dataset.keys(), axes)):
        ax.plot(dataset[case][policy]["data"]["pos_x"], 
                dataset[case][policy]["data"]["pos_y"])
        
        # Set x and y limits for the plot
        ax.set_xlim([-0.5, 3.5])
        ax.set_ylim([-2, 2])
        
        # Set title for each subplot
        ax.set_title(f"{case}/{policy}")
        
        # Optionally, set x and y axis labels
        ax.set_xlabel("X Position")
        ax.set_ylabel("Y Position")
    
    # Remove any extra subplots if the number of cases is not a perfect square
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
    
    # Adjust layout to avoid overlap
    plt.tight_layout()
    plt.show()

In [ ]:
plot_trajectory(selected_dataset , policy=0)

In [ ]:
plot_performance(performance, title="reward")
plot_performance(avg_vel , title="average velocity")

## PCA

Run Visualize th pca result

In [ ]:
def analyze_weights_pca(dataset, max_components=9, variance_threshold=0.95, env_idx=0, plot=True , suppress_print=False):
    """
    Perform PCA analysis on neural network weights from dataset
    
    Parameters:
    -----------
    dataset : dict
        Your dataset containing brain weights (selected_dataset["full"][0])
    max_components : int, default=9
        Maximum number of PCA components to analyze
    variance_threshold : float, default=0.95
        Threshold for finding minimum components (e.g., 0.95 = 95%)
    env_idx : int, default=0
        Which environment index to analyze (for population-based training)
    plot : bool, default=True
        Whether to generate plots
        
    Returns:
    --------
    dict : Contains PCA results and explained variance for each layer
    """

    # Initialize storage
    pca_results = {}
    explained_variance = {}
    
    # Get layer keys
    layer_keys = list(dataset["brain"]["weight"].keys())
    print(f"Found {len(layer_keys)} layers: {layer_keys[:3]}...")  # Show first 3
    
    for layer_name in layer_keys:
        if layer_name not in ['layer0', 'layer1', 'layer2']:
            continue  # Skip if more than expected layers
            
        print(f"\nAnalyzing {layer_name}...")
        
        # Get data for current layer
        layer_data = dataset["brain"]["weight"][layer_name]
        
        # Reshape data: (timesteps, flattened_weights)
        data = layer_data[:, env_idx].reshape(layer_data.shape[0], -1)
        print(f"Data shape for PCA: {data.shape}")
        
        # Store results for this layer
        pca_results[layer_name] = {}
        explained_variance[layer_name] = {}
        
        # Determine max components (can't exceed data dimensions)
        max_comp = min(max_components, data.shape[0], data.shape[1])
        
        for num_components in range(1, max_comp + 1):
            pca = PCA(n_components=num_components)
            transformed_data = pca.fit_transform(data)
            
            # Store results
            pca_results[layer_name][num_components] = {
                'pca_model': pca,
                'transformed_data': transformed_data
            }
            
            # Store explained variance information
            explained_variance[layer_name][num_components] = {
                'explained_variance_ratio': pca.explained_variance_ratio_,
                'cumulative_variance_ratio': np.cumsum(pca.explained_variance_ratio_),
                'total_variance_explained': np.sum(pca.explained_variance_ratio_),
                'singular_values': pca.singular_values_,
                'explained_variance': pca.explained_variance_
            }
            
        # Print summary for this layer
        print(f"  Component analysis summary:")
        for n in [1, 2, 3, 5] if max_comp >= 5 else range(1, max_comp + 1):
            if n <= max_comp:
                total_var = explained_variance[layer_name][n]['total_variance_explained']
                print(f"    {n} components: {total_var:.4f} ({total_var*100:.2f}%)")
    
    # Find components needed for variance threshold
    print(f"\nComponents needed for {variance_threshold*100}% variance:")
    threshold_results = {}
    
    for layer_name in explained_variance:
        for n_comp in explained_variance[layer_name]:
            cum_var = explained_variance[layer_name][n_comp]['total_variance_explained']
            if cum_var >= variance_threshold:
                threshold_results[layer_name] = (n_comp, cum_var)
                print(f"  {layer_name}: {n_comp} components ({cum_var:.4f})")
                break
        else:
            threshold_results[layer_name] = (max_comp, explained_variance[layer_name][max_comp]['total_variance_explained'])
            print(f"  {layer_name}: >{max_comp} components needed (max tested: {explained_variance[layer_name][max_comp]['total_variance_explained']:.4f})")
    
    # Generate plots if requested
    if plot:
        _plot_pca_results(explained_variance, variance_threshold)

    # Return complete results
    return {
        'pca_results': pca_results,
        'explained_variance': explained_variance,
        'threshold_results': threshold_results,
        'summary': {
            'layers_analyzed': list(explained_variance.keys()),
            'max_components_tested': max_components,
            'variance_threshold': variance_threshold,
            'environment_index': env_idx
        }
    }

def _plot_pca_results(explained_variance, threshold=0.95):
    """Helper function to plot PCA results"""
    
    n_layers = len(explained_variance)
    fig, axes = plt.subplots(2, n_layers, figsize=(5*n_layers, 8))
    
    if n_layers == 1:
        axes = axes.reshape(2, 1)
    
    for idx, layer_name in enumerate(explained_variance.keys()):
        components = list(explained_variance[layer_name].keys())
        
        # Individual explained variance (first component only)
        individual_var = [explained_variance[layer_name][n]['explained_variance_ratio'][0] 
                         for n in components]
        
        # Cumulative explained variance
        cumulative_var = [explained_variance[layer_name][n]['total_variance_explained'] 
                         for n in components]
        
        # Plot individual variance
        axes[0, idx].bar(components, individual_var)
        axes[0, idx].set_xlabel('Principal Component')
        axes[0, idx].set_ylabel('Explained Variance Ratio')
        axes[0, idx].set_title(f'{layer_name} - First Component Variance')
        axes[0, idx].grid(True, alpha=0.3)
        
        # Plot cumulative variance
        axes[1, idx].plot(components, cumulative_var, 'bo-')
        axes[1, idx].set_xlabel('Number of Components')
        axes[1, idx].set_ylabel('Cumulative Explained Variance')
        axes[1, idx].set_title(f'{layer_name} - Cumulative Variance')
        axes[1, idx].grid(True, alpha=0.3)
        axes[1, idx].axhline(y=threshold, color='r', linestyle='--', alpha=0.7, 
                           label=f'{threshold*100}% threshold')
        axes[1, idx].legend()
    
    plt.tight_layout()
    plt.show()


In [ ]:
skip = True
if not skip:
    results = analyze_weights_pca(selected_dataset["full"][0] , suppress_print=True)
    layer0_pca = results['explained_variance']['layer0'][2]['explained_variance_ratio']
    threshold_info = results['threshold_results']['layer0']

In [ ]:
pos_idx = np.arange(0,19)
vel_idx = np.arange(19,19*2)
action_idx = np.arange(19*2,19*3)
IMU_idx = np.arange(57,60)
fc_idx = np.arange(60,64)
group = [pos_idx , vel_idx , action_idx , IMU_idx , fc_idx]

In [ ]:
import numpy as np
from scipy.spatial.distance import cdist
from scipy.special import psi

def ksg_estimator(X, Y, k):
    N = len(X)
    
    # Combine X and Y into a joint dataset Z
    Z = np.column_stack((X, Y))
    
    # Calculate pairwise distances in the joint space and in the marginal spaces
    dist_Z = cdist(Z, Z, metric='euclidean')
    dist_X = cdist(X, X, metric='euclidean')
    dist_Y = cdist(Y, Y, metric='euclidean')

    # Sort the distances and find the k-th nearest neighbor distances
    dist_Z_sorted = np.sort(dist_Z, axis=1)
    dist_X_sorted = np.sort(dist_X, axis=1)
    dist_Y_sorted = np.sort(dist_Y, axis=1)
    # Get the k-th neighbor distances for joint and marginal spaces
    eps_Z = dist_Z_sorted[:, k]
    eps_X = dist_X_sorted[:, k]
    eps_Y = dist_Y_sorted[:, k]
    eps_max = np.maximum(eps_X, eps_Y)
    
    # Count the number of neighbors within the k-th neighbor distance for X and Y
    nx = np.array([np.sum(dist_X[i] <= eps_max[i]) for i in range(N)])
    ny = np.array([np.sum(dist_Y[i] <= eps_max[i]) for i in range(N)])
    
    # Compute the digamma functions and the final MI estimate
    digamma_k = psi(k)
    digamma_N = psi(N)
    
    # Average over all data points
    avg_digamma_nx_ny = np.mean(psi(nx + 1) + psi(ny + 1))
    
    # MI estimate
    MI = digamma_k - avg_digamma_nx_ny + digamma_N
    return MI

# Example usage with random data
# X = np.random.rand(1000,3)
# Y = np.random.rand(1000,3)
# k = 5  # Number of neighbors

# mi = ksg_estimator(X, Y, k)
# print(f"Estimated MI: {mi}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression

def MI_PCA_InputSensor_Hidden(dataset , pc_components , k , group,mode):
    '''
    Return : [sensor , n component , n compare layer , MI of each component]
    '''
    out = []
    for index, sensory in enumerate(group):
        out.append(MI_sensor_to_hidden(dataset , pc_components , k ,sensory,mode))
        
    return np.average(out,axis=1)

def MI_sensor_to_hidden(dataset, robot_idx, pc_components, k, group, mode):
    """
    Modified to handle robot index for multi-environment data
    """
    temp = []
    # Extract data for specific robot
    layer0 = PCA(n_components=pc_components).fit_transform(
        dataset["brain"]["weight"]["layer0"][:, robot_idx, group, :].reshape(
            dataset["brain"]["weight"]["layer0"][:, robot_idx, group, :].shape[0], -1
        )
    )
    layer1 = PCA(n_components=pc_components).fit_transform(
        dataset["brain"]["weight"]["layer1"][:, robot_idx].reshape(
            dataset["brain"]["weight"]["layer1"][:, robot_idx].shape[0], -1
        )
    )
    layer2 = PCA(n_components=pc_components).fit_transform(
        dataset["brain"]["weight"]["layer2"][:, robot_idx].reshape(
            dataset["brain"]["weight"]["layer2"][:, robot_idx].shape[0], -1
        )
    )
    
    for i in range(pc_components):
        if mode == "avg":
            a = np.average(mutual_info_regression(layer1, layer0[:, i], n_neighbors=k))
            b = np.average(mutual_info_regression(layer2, layer0[:, i], n_neighbors=k))
        elif mode == "ksg":
            a = ksg_estimator(layer1, layer0, k=k)
            b = ksg_estimator(layer2, layer0, k=k)
        temp.append(np.array([a, b]))
    
    return np.array(temp)  # Shape: [pc_components, 2]

def MI_PCA_flexible_comparison(dataset, comparison_type, target_case=None, target_policy=None, 
                              comparison_items=None, num_env=10, pc_components=3, k=4, group=[0,1,2,3,4], mode="avg"):
    """
    Flexible MI analysis for different comparison types
    
    Parameters:
    - dataset: Dictionary with structure dataset[case][policy]
    - comparison_type: "policies" or "cases"
    - target_case: Case name when comparing policies (required if comparison_type="policies")
    - target_policy: Policy index when comparing cases (required if comparison_type="cases")
    - comparison_items: List of policies or cases to compare (if None, use all available)
    - num_env: Number of environments
    - pc_components: Number of principal components
    - k: Number of neighbors for MI estimation
    - group: List of sensor indices
    - mode: "avg" or "ksg"
    
    Returns:
    - results: Dictionary with comparison results
    - organized_data: Array for plotting
    - comparison_labels: Labels for the compared items
    """
    
    if comparison_type == "policies":
        if target_case is None:
            raise ValueError("target_case must be specified when comparing policies")
        
        available_policies = list(range(len(dataset[target_case])))
        if comparison_items is None:
            comparison_items = available_policies
        else:
            comparison_items = [p for p in comparison_items if p in available_policies]
        
        comparison_labels = [f"Policy {p}" for p in comparison_items]
        print(f"Comparing policies {comparison_items} in case '{target_case}'")
        
    elif comparison_type == "cases":
        if target_policy is None:
            raise ValueError("target_policy must be specified when comparing cases")
        
        available_cases = list(dataset.keys())
        if comparison_items is None:
            comparison_items = available_cases
        else:
            comparison_items = [c for c in comparison_items if c in available_cases]
        
        comparison_labels = [f"Case {c}" for c in comparison_items]
        print(f"Comparing cases {comparison_items} with policy {target_policy}")
        
    else:
        raise ValueError("comparison_type must be 'policies' or 'cases'")
    
    # Initialize storage
    num_items = len(comparison_items)
    num_sensors = len(group)
    num_layers = 2
    
    results = {}
    # Shape: [n_items, n_sensors, n_layers, n_envs, n_components]
    organized_data = np.zeros((num_items, num_sensors, num_layers, num_env, pc_components))
    
    print(f"Processing {num_items} items, {num_env} environments, {num_sensors} sensors...")
    
    for item_idx, item in enumerate(comparison_items):
        if comparison_type == "policies":
            current_data = dataset[target_case][item]
            print(f"Processing policy {item} in case {target_case}")
        else:  # cases
            current_data = dataset[item][target_policy]
            print(f"Processing case {item} with policy {target_policy}")
        
        item_data = []
        
        for robot in range(num_env):
            # Get MI data for this robot: [n_sensors, n_components, n_layers]
            robot_mi = []
            for sensor_idx, sensor in enumerate(group):
                sensor_mi = MI_sensor_to_hidden(current_data, robot, pc_components, k, [sensor], mode)
                robot_mi.append(sensor_mi)  # Shape: [pc_components, 2]
            
            robot_mi = np.array(robot_mi)  # Shape: [n_sensors, pc_components, 2]
            item_data.append(robot_mi)
        
        item_data = np.array(item_data)  # Shape: [n_envs, n_sensors, pc_components, 2]
        results[item] = item_data
        
        # Reorganize for plotting: [n_sensors, n_layers, n_envs, n_components]
        for sensor_i in range(num_sensors):
            for layer_i in range(num_layers):
                for env_i in range(num_env):
                    organized_data[item_idx, sensor_i, layer_i, env_i, :] = item_data[env_i, sensor_i, :, layer_i]
    
    return results, organized_data, comparison_labels

def plot_MI_boxplots_comparison(organized_data, group, comparison_labels, show_scatter=True, figsize=(20, 15) , label=None):
    """
    Create boxplots for MI comparison data with optional scatter points overlay
    """
    n_items, n_sensors, n_layers, n_envs, n_components = organized_data.shape
    
    # Create subplots: one row per comparison item
    fig, axes = plt.subplots(n_items, 1, figsize=figsize, squeeze=False)
    
    colors = ['lightblue', 'lightcoral']
    scatter_colors = ['darkblue', 'darkred']
    layer_names = ['Hidden Layer 1', 'Hidden Layer 2']
    
    for item_idx, item_label in enumerate(comparison_labels):
        ax = axes[item_idx, 0]
        
        # Prepare data for boxplot
        boxplot_data = []
        labels = []
        positions = []
        colors_list = []
        
        pos = 1
        for sensor_i in range(n_sensors):
            for layer_i in range(n_layers):
                # Flatten across environments and components: [n_envs * n_components]
                data = organized_data[item_idx, sensor_i, layer_i, :, :].flatten()
                boxplot_data.append(data)
                if label==None:
                    labels.append(f'S{sensor_i}-L{layer_i+1}')
                elif label != None:
                    labels.append(f"{label[sensor_i]}-L{layer_i+1}")
                positions.append(pos)
                colors_list.append(colors[layer_i])
                pos += 1
        
        # Create boxplot
        bp = ax.boxplot(boxplot_data, positions=positions, patch_artist=True, widths=0.6)
        
        # Color the boxes
        for patch, color in zip(bp['boxes'], colors_list):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        # Add scatter points on top of boxplots
        if show_scatter:
            for i, data in enumerate(boxplot_data):
                x_pos = positions[i]
                # Add small random jitter to x positions for better visibility
                x_jitter = np.random.normal(x_pos, 0.05, size=len(data))
                
                # Use different colors for different layers
                layer_idx = i % 2  # 0 for Layer 1, 1 for Layer 2
                ax.scatter(x_jitter, data, 
                          color=scatter_colors[layer_idx], 
                          alpha=0.6, 
                          s=15,
                          edgecolors='white',
                          linewidth=0.5,
                          zorder=5)  # Ensure points are on top
        
        # Customize plot
        ax.set_title(f'{item_label} (Envs: {n_envs}, Components: {n_components})')
        ax.set_xlabel('Sensors and Hidden Layers')
        ax.set_ylabel('Mutual Information')
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=45)
        ax.grid(True, alpha=0.3)
        
        # Add statistics (move slightly lower when scatter is shown)
        text_y_pos = 0.90 if show_scatter else 0.95
        for i, data in enumerate(boxplot_data):
            mean_val = np.mean(data)
            std_val = np.std(data)
            ax.text(positions[i], ax.get_ylim()[1] * text_y_pos, 
                   f'{mean_val:.3f}\n±{std_val:.3f}', 
                   ha='center', va='top', fontsize=6,
                   bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
        
        # Add vertical lines to separate sensors
        for sensor_i in range(1, n_sensors):
            separator_pos = sensor_i * n_layers + 0.5
            ax.axvline(x=separator_pos, color='gray', linestyle='--', alpha=0.5)
        
        # Add legend for scatter colors (only for first subplot)
        if show_scatter and item_idx == 0:
            legend_elements = [
                plt.scatter([], [], color=scatter_colors[0], s=30, label='Layer 1 Points'),
                plt.scatter([], [], color=scatter_colors[1], s=30, label='Layer 2 Points')
            ]
            ax.legend(handles=legend_elements, loc='upper right', fontsize=8)
    
    scatter_text = " with Individual Points" if show_scatter else ""
    plt.suptitle(f'Mutual Information Comparison{scatter_text}: {comparison_labels[0].split()[0]}s\n'
                f'Sensors: {group}, Environments: {n_envs}')
    plt.tight_layout()
    plt.show()
    
    return fig

# Usage functions
def run_policy_comparison(dataset, target_case, policies_to_compare=None, num_env=10, 
                         pc_components=3, k=4, group=[0,1,2,3,4], mode="avg", show_scatter=True , label=None):
    """
    Compare different policies within the same case
    
    Example:
    results, data, fig = run_policy_comparison(
        selected_dataset, 
        target_case="full",
        policies_to_compare=[0, 1, 2],  # Compare policies 0, 1, 2
        num_env=10,
        pc_components=3,
        k=4,
        group=[0,1,2,3,4],
        mode="avg",
        show_scatter=True
    )
    """
    print(f"Comparing policies in case '{target_case}'")
    
    results, organized_data, comparison_labels = MI_PCA_flexible_comparison(
        dataset=dataset,
        comparison_type="policies",
        target_case=target_case,
        comparison_items=policies_to_compare,
        num_env=num_env,
        pc_components=pc_components,
        k=k,
        group=group,
        mode=mode
    )
    
    fig = plot_MI_boxplots_comparison(
        organized_data, group, comparison_labels, show_scatter=show_scatter,label=label
    )
    
    print(f"Analysis complete! Compared {len(comparison_labels)} policies in case '{target_case}'")
    return results, organized_data, fig

def run_case_comparison(dataset, target_policy, cases_to_compare=None, num_env=10,
                       pc_components=3, k=4, group=[0,1,2,3,4], mode="avg", show_scatter=True):
    """
    Compare different cases with the same policy
    
    Example:
    results, data, fig = run_case_comparison(
        selected_dataset, 
        target_policy=2,
        cases_to_compare=["full", "partial", "test"],  # Compare different cases
        num_env=10,
        pc_components=3,
        k=4,
        group=[0,1,2,3,4],
        mode="avg",
        show_scatter=True
    )
    """
    print(f"Comparing cases with policy {target_policy}")
    
    results, organized_data, comparison_labels = MI_PCA_flexible_comparison(
        dataset=dataset,
        comparison_type="cases",
        target_policy=target_policy,
        comparison_items=cases_to_compare,
        num_env=num_env,
        pc_components=pc_components,
        k=k,
        group=group,
        mode=mode
    )
    
    fig = plot_MI_boxplots_comparison(
        organized_data, group, comparison_labels, show_scatter=show_scatter
    )
    
    print(f"Analysis complete! Compared {len(comparison_labels)} cases with policy {target_policy}")
    return results, organized_data, fig

def plot_comparison_heatmap(organized_data, group, comparison_labels, figsize=(12, 8)):
    """
    Create a heatmap comparing average MI across items, sensors, and layers
    """
    n_items, n_sensors, n_layers, n_envs, n_components = organized_data.shape
    
    # Average across environments and components
    avg_data = np.mean(organized_data, axis=(3, 4))  # Shape: [n_items, n_sensors, n_layers]
    
    # Reshape for heatmap: [n_items, n_sensors * n_layers]
    heatmap_data = avg_data.reshape(n_items, n_sensors * n_layers)
    
    # Create labels
    labels = []
    for sensor_i in range(n_sensors):
        for layer_i in range(n_layers):
            if labels is not None:
                labels.append(f'S{group[sensor_i]}-L{layer_i+1}')
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(heatmap_data, cmap='viridis', aspect='auto')
    
    # Customize
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45)
    ax.set_yticks(range(n_items))
    ax.set_yticklabels([label.replace('Case ', '').replace('Policy ', 'P') for label in comparison_labels])
    ax.set_xlabel('Sensors and Hidden Layers')
    ax.set_ylabel('Comparison Items')
    ax.set_title('Average Mutual Information Heatmap')
    
    # Add colorbar
    plt.colorbar(im, ax=ax, label='Mutual Information')
    
    # Add text annotations
    for item_i in range(n_items):
        for pos_i in range(len(labels)):
            text = ax.text(pos_i, item_i, f'{heatmap_data[item_i, pos_i]:.3f}',
                         ha="center", va="center", color="white", fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Example usage patterns:

# 1. Compare policies within a case (with scatter points)
# results, data, fig = run_policy_comparison(
#     selected_dataset, 
#     target_case="full",
#     policies_to_compare=[0, 1, 2, 3],
#     num_env=10,
#     show_scatter=True
# )

# 2. Compare cases with same policy (with scatter points)
# results, data, fig = run_case_comparison(
#     selected_dataset, 
#     target_policy=2,
#     cases_to_compare=["full", "partial"],
#     num_env=10,
#     show_scatter=True
# )

# 3. Compare policies within a case (boxplot only)
# results, data, fig = run_policy_comparison(
#     selected_dataset, 
#     target_case="full",
#     policies_to_compare=[0, 1, 2, 3],
#     num_env=10,
#     show_scatter=False  # No scatter points
# )

# 4. Create heatmap for quick overview
# results, data, fig_box = run_case_comparison(selected_dataset, target_policy=2, cases_to_compare=["full", "partial"])
# fig_heatmap = plot_comparison_heatmap(data, group=[0,1,2,3,4], comparison_labels=["Case full", "Case partial"])

In [ ]:
import numpy as np
import pandas as pd

def rank_mi_per_policy(organized_data, sensor_labels, policy_ids, top_k=None):
    """
    organized_data: [n_items, n_sensors, n_layers, n_envs, n_components]
    sensor_labels:  list of sensor/joint names (len = n_sensors)
    policy_ids:     the list you passed (e.g., [0,1,2]) in that order
    top_k:          if set, return only top_k per layer
    """
    # Average over environments & PCA components -> [n_items, n_sensors, n_layers]
    avg = organized_data.mean(axis=(3,4))

    rankings = {}
    for item_idx, policy in enumerate(policy_ids):
        rankings[policy] = {}
        for layer_i in range(avg.shape[2]):  # 0 -> L1, 1 -> L2
            scores = avg[item_idx, :, layer_i]
            order = np.argsort(-scores)  # descending
            entries = [(sensor_labels[s], float(scores[s])) for s in order]
            if top_k is not None:
                entries = entries[:top_k]
            rankings[policy][f"L{layer_i+1}"] = entries
    return rankings

def rankings_to_dataframe(rankings, case_name):
    """
    Convert the dict from rank_mi_per_policy into a tidy DataFrame:
    columns: [case, policy, layer, rank, sensor, mi]
    """
    rows = []
    for policy, layer_dict in rankings.items():
        for layer, items in layer_dict.items():
            for r, (sensor, mi) in enumerate(items, start=1):
                rows.append((case_name, policy, layer, r, sensor, mi))
    return pd.DataFrame(rows, columns=["case","policy","layer","rank","sensor","mi"])


In [ ]:
# 1. Compare policies within a case (with scatter points)
group = [[i,i+19,i+19*2] for i in range(19)]
label = ['joint0_m', 'joint0_h', 'joint0_f', 'joint1_rf', 'joint1_lf', 'joint1_rh', 'joint1_lh', 'joint2_rf', 'joint2_lf', 'joint2_rh', 'joint2_lh', 'joint3_rf', 'joint3_lf', 'joint3_rh', 'joint3_lh', 'joint4_rf', 'joint4_lf', 'joint4_rh', 'joint4_lh']

print("\n=== Creating Heatmap for Policy Comparison ===")
layer_names = ["L1" , "L2"]
results, data, fig = run_policy_comparison(
    selected_dataset, 
    target_case="flat",
    policies_to_compare=[0, 1, 2],
    num_env=30,
    show_scatter=True,
    group=group ,
    mode="avg",       # Add missing parameter
)
results, data, fig = run_policy_comparison(
    selected_dataset, 
    target_case="rough",
    policies_to_compare=[0, 1, 2],
    num_env=30,
    show_scatter=True,
    group=group ,
    mode="avg",       # Add missing parameter
)
results, data, fig = run_policy_comparison(
    selected_dataset, 
    target_case="weight",
    policies_to_compare=[0, 1, 2],
    num_env=30,
    show_scatter=True,
    group=group ,
    mode="avg",       # Add missing parameter
)


# 4. Create heatmap for the policy comparison

# comparison_labels = ["Policy 0", "Policy 1", "Policy 2"]  # Match your actual policies
# fig_heatmap = plot_comparison_heatmap(
#     data, 
#     group=group, 
#     comparison_labels=comparison_labels,
#     sensor_names=sensor_names,
#     layer_names=layer_names,
# )

In [ ]:
results, data, fig = run_policy_comparison(
    selected_dataset, 
    target_case="flat",
    policies_to_compare=[0, 1, 2],
    num_env=30,
    show_scatter=True,
    group=group ,
    mode="ksg",       # Add missing parameter
)
results, data, fig = run_policy_comparison(
    selected_dataset, 
    target_case="rough",
    policies_to_compare=[0, 1, 2],
    num_env=30,
    show_scatter=True,
    group=group ,
    mode="ksg",       # Add missing parameter
)
results, data, fig = run_policy_comparison(
    selected_dataset, 
    target_case="weight",
    policies_to_compare=[0, 1, 2],
    num_env=30,
    show_scatter=True,
    group=group ,
    mode="ksg",       # Add missing parameter
)


In [ ]:
import numpy as np
import pandas as pd

def rank_two_layers_per_policy(organized_data, sensor_labels, policy_ids, top_k=None):
    """
    Make two rankings (L1-only and L2-only) per policy.
    
    Inputs
    -------
    organized_data : np.ndarray
        Shape [n_items, n_sensors, n_layers(=2), n_envs, n_components]
        (This is exactly what run_policy_comparison returns)
    sensor_labels : list[str]
        Names for each sensor/joint, length = n_sensors
    policy_ids : list[int]
        Policies you compared (order must match policies_to_compare)
    top_k : Optional[int]
        If given, truncate to top_k rows per layer; else return full ranking
    
    Returns
    -------
    rankings : dict
        {
          policy_id: {
             "L1": [(sensor_name, mi), ... highest->lowest],
             "L2": [(sensor_name, mi), ... highest->lowest],
          },
          ...
        }
    """
    # Average across envs & components -> [n_items, n_sensors, n_layers]
    avg = np.nanmean(organized_data, axis=(3, 4))

    n_items, n_sensors, n_layers = avg.shape
    assert n_layers == 2, "Expected exactly 2 hidden layers in organized_data."

    # Fallback labels if length mismatch
    if sensor_labels is None or len(sensor_labels) != n_sensors:
        sensor_labels = [f"S{i}" for i in range(n_sensors)]

    rankings = {}
    for item_idx, policy in enumerate(policy_ids):
        rankings[policy] = {}
        # L1 (layer_i = 0)
        scores_L1 = avg[item_idx, :, 0]
        order_L1 = np.argsort(-scores_L1)
        list_L1 = [(sensor_labels[s], float(scores_L1[s])) for s in order_L1]
        if top_k is not None:
            list_L1 = list_L1[:top_k]
        rankings[policy]["L1"] = list_L1

        # L2 (layer_i = 1)
        scores_L2 = avg[item_idx, :, 1]
        order_L2 = np.argsort(-scores_L2)
        list_L2 = [(sensor_labels[s], float(scores_L2[s])) for s in order_L2]
        if top_k is not None:
            list_L2 = list_L2[:top_k]
        rankings[policy]["L2"] = list_L2

    return rankings

def rankings_layer_to_dataframe(rankings, case_name, layer="L1"):
    """
    Convert ONE layer's rankings per policy to a tidy DataFrame.
    Columns: case, policy, layer, rank, sensor, mi
    """
    rows = []
    for policy, layer_dict in rankings.items():
        items = layer_dict[layer]
        for r, (sensor, mi) in enumerate(items, start=1):
            rows.append((case_name, policy, layer, r, sensor, mi))
    return pd.DataFrame(rows, columns=["case", "policy", "layer", "rank", "sensor", "mi"])

def print_two_layer_rankings(rankings, case_name, top_k_preview=10):
    """
    Nicely print top-K preview for both layers per policy.
    """
    print(f"\n=== {case_name.upper()} : Two-layer Rankings per Policy ===")
    for policy in sorted(rankings.keys()):
        print(f"\nPolicy {policy}")
        for L in ("L1", "L2"):
            print(f"  {L} top sensors:")
            for i, (sensor, mi) in enumerate(rankings[policy][L][:top_k_preview], start=1):
                print(f"    {i:>2}. {sensor:12s}  MI={mi:.5f}")


In [ ]:
# 1. Compare policies within a case (with scatter points)
label = ['joint0_m', 'joint0_h', 'joint0_f', 'joint1_rf', 'joint1_lf', 'joint1_rh', 'joint1_lh', 'joint2_rf', 'joint2_lf', 'joint2_rh', 'joint2_lh', 'joint3_rf', 'joint3_lf', 'joint3_rh', 'joint3_lh', 'joint4_rf', 'joint4_lf', 'joint4_rh', 'joint4_lh']
group = [[0*i , 19+i , 2*19+i] for i in range(19)]
print("\n=== Creating Heatmap for Policy Comparison ===")
layer_names = ["L1" , "L2"]
policies = [0,1,2]

# --- flat ---
res_flat, data_flat, _ = run_policy_comparison(
    selected_dataset, target_case="flat",
    policies_to_compare=policies, num_env=30,
    show_scatter=True, group=group, mode="ksg", label=label
)
rank_flat = rank_two_layers_per_policy(data_flat, label, policies, top_k=None)  # full sorted lists
print_two_layer_rankings(rank_flat, "flat", top_k_preview=5)

# If you want DataFrames separated by layer:
df_flat_L1 = rankings_layer_to_dataframe(rank_flat, "flat", layer="L1")
df_flat_L2 = rankings_layer_to_dataframe(rank_flat, "flat", layer="L2")

# --- rough ---
res_rough, data_rough, _ = run_policy_comparison(
    selected_dataset, target_case="rough",
    policies_to_compare=policies, num_env=30,
    show_scatter=True, group=group, mode="ksg", label=label
)
rank_rough = rank_two_layers_per_policy(data_rough, label, policies)
print_two_layer_rankings(rank_rough, "rough", top_k_preview=5)
df_rough_L1 = rankings_layer_to_dataframe(rank_rough, "rough", layer="L1")
df_rough_L2 = rankings_layer_to_dataframe(rank_rough, "rough", layer="L2")

# --- weight ---
res_weight, data_weight, _ = run_policy_comparison(
    selected_dataset, target_case="weight",
    policies_to_compare=policies, num_env=30,
    show_scatter=True, group=group, mode="ksg", label=label
)
rank_weight = rank_two_layers_per_policy(data_weight, label, policies)
print_two_layer_rankings(rank_weight, "weight", top_k_preview=5)
df_weight_L1 = rankings_layer_to_dataframe(rank_weight, "weight", layer="L1")
df_weight_L2 = rankings_layer_to_dataframe(rank_weight, "weight", layer="L2")

In [ ]:
import re
import numpy as np
import pandas as pd

def rank_motors_from_df(df, group_regex=r'^(joint\d+)', top_k=None):
    """
    Rank motors per policy by mean MI, grouping sensors like 'joint1_lf', 'joint1_rf' -> 'joint1'.

    Parameters
    ----------
    df : DataFrame with columns ['case','policy','layer','rank','sensor','mi']
         (Here you’ll pass df_flat_L1 for L1, or df_flat_L2 for L2.)
    group_regex : regex to extract motor group from 'sensor' (default: '^(joint\\d+)')
    top_k : optional int, if set, keep only top_k motors per policy

    Returns
    -------
    ranked : DataFrame with columns ['case','policy','layer','motor','mi_mean','rank']
             ranked descending by mi_mean within each (case, policy, layer)
    """
    work = df.copy()

    # 1) Extract motor group from sensor name
    work['motor'] = work['sensor'].str.extract(group_regex)
    work = work.dropna(subset=['motor'])

    # 2) Aggregate MI by (case, policy, layer, motor)
    agg = (work.groupby(['case','policy','layer','motor'], as_index=False)['mi']
                .mean()
                .rename(columns={'mi':'mi_mean'}))

    # 3) Rank within each (case, policy, layer)
    agg['rank'] = (agg.groupby(['case','policy','layer'])['mi_mean']
                     .rank(method='first', ascending=False).astype(int))

    # 4) Sort for readability
    ranked = agg.sort_values(['case','policy','layer','mi_mean'], ascending=[True, True, True, False])

    # 5) Optional: keep top_k per (case, policy, layer)
    if top_k is not None:
        ranked = (ranked.groupby(['case','policy','layer'], group_keys=False)
                        .head(top_k))

    return ranked

# --- Use with your df_flat_L1 (Hidden Layer 1) ---
rank_flat_L1 = rank_motors_from_df(df_flat_L1, top_k=None)   # full list per policy
# Example: show top-5 motors for each policy in L1
print(rank_flat_L1.groupby(['policy']).head(6)[['policy','motor','mi_mean','rank']])

# --- Use with your df_flat_L1 (Hidden Layer 1) ---
rank_flat_L2 = rank_motors_from_df(df_flat_L2, top_k=None)   # full list per policy
# Example: show top-5 motors for each policy in L1
print(rank_flat_L2.groupby(['policy']).head(6)[['policy','motor','mi_mean','rank']])

In [ ]:
def heatmap(data, row_labels, col_labels, ax=None,
            cbar_kw=None, cbarlabel="" ,cbar=True, **kwargs):
    """
    Create a heatmap from a numpy array and two lists of labels.

    Parameters
    ----------
    data
        A 2D numpy array of shape (M, N).
    row_labels
        A list or array of length M with the labels for the rows.
    col_labels
        A list or array of length N with the labels for the columns.
    ax
        A `matplotlib.axes.Axes` instance to which the heatmap is plotted.  If
        not provided, use current Axes or create a new one.  Optional.
    cbar_kw
        A dictionary with arguments to `matplotlib.Figure.colorbar`.  Optional.
    cbarlabel
        The label for the colorbar.  Optional.
    **kwargs
        All other arguments are forwarded to `imshow`.
    """

    if ax is None:
        ax = plt.gca()

    if cbar_kw is None:
        cbar_kw = {}

    # Plot the heatmap
    im = ax.imshow(data, **kwargs)

    # Create colorbar
    if cbar:
        cbar = ax.figure.colorbar(im, ax=ax, **cbar_kw)
        cbar.ax.set_ylabel(cbarlabel, rotation=-90, va="bottom")

    # Show all ticks and label them with the respective list entries.
    ax.set_xticks(range(data.shape[1]), labels=col_labels,
                  rotation=45, ha="right", rotation_mode="anchor")
    ax.set_yticks(range(data.shape[0]), labels=row_labels)

    # Let the horizontal axes labeling appear on top.
    # ax.tick_params(top=True, bottom=False,
    #                labeltop=True, labelbottom=False)

    # Turn spines off and create white grid.
    ax.spines[:].set_visible(False)

    ax.set_xticks(np.arange(data.shape[1]+1)-.5, minor=True)
    ax.set_yticks(np.arange(data.shape[0]+1)-.5, minor=True)
    ax.grid(which="minor", color="w", linestyle='-', linewidth=3)
    ax.tick_params(which="minor", bottom=False, left=False)

    return im, cbar


def annotate_heatmap(im, data=None, valfmt="{x:.2f}",
                     textcolors=("black", "white"),
                     threshold=None, **textkw):
    """
    A function to annotate a heatmap.

    Parameters
    ----------
    im
        The AxesImage to be labeled.
    data
        Data used to annotate.  If None, the image's data is used.  Optional.
    valfmt
        The format of the annotations inside the heatmap.  This should either
        use the string format method, e.g. "$ {x:.2f}", or be a
        `matplotlib.ticker.Formatter`.  Optional.
    textcolors
        A pair of colors.  The first is used for values below a threshold,
        the second for those above.  Optional.
    threshold
        Value in data units according to which the colors from textcolors are
        applied.  If None (the default) uses the middle of the colormap as
        separation.  Optional.
    **kwargs
        All other arguments are forwarded to each call to `text` used to create
        the text labels.
    """

    if not isinstance(data, (list, np.ndarray)):
        data = im.get_array()

    # Normalize the threshold to the images color range.
    if threshold is not None:
        threshold = im.norm(threshold)
    else:
        threshold = im.norm(data.max())/2.

    # Set default alignment to center, but allow it to be
    # overwritten by textkw.
    kw = dict(horizontalalignment="center",
              verticalalignment="center")
    kw.update(textkw)

    # Get the formatter in case a string is supplied
    if isinstance(valfmt, str):
        valfmt = StrMethodFormatter(valfmt)

    # Loop over the data and create a `Text` for each "pixel".
    # Change the text's color depending on the data.
    texts = []
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            kw.update(color=textcolors[int(im.norm(data[i, j]) > threshold)])
            text = im.axes.text(j, i, valfmt(data[i, j], None), **kw)
            texts.append(text)

    return texts

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 8))
col_label = ['layer2', 'layer3']
im , cbar = heatmap(out1, row_labels=['pos', 'vel', 'action', 'IMU', 'fc'], col_labels=col_label, cbarlabel="MI (nats)", cmap="YlGn" , cbar=False)
_ = annotate_heatmap(im=im , valfmt="{x:.2f}")
fig.tight_layout()